# 04 — LOSO Site-Level Evaluation

Site-level statistical analyses that are unique to Leave-One-Site-Out cross-validation.
Pooled LOSO metrics (with sample-level bootstrap CIs) are handled by `02_evaluate.ipynb`
via `predictions/loso/*.csv`. This notebook adds:

- **Cluster bootstrap CIs** — resampling is over the 20 sites (the unit of spatial generalisation),
  not individual samples. This gives the methodologically correct uncertainty estimate for LOSO.
- **Site-paired permutation test** — tests whether model pairs differ significantly across sites
  (n=20 paired per-site RMSE values).
- **Per-site forest plot** — per-site RMSE ± cluster-bootstrap CI for each model.

**Prerequisite:** run `03_loso.ipynb` first to produce `results/loso/per_site_metrics.csv`
and `results/loso/loso_predictions.csv`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from itertools import combinations
from scipy import stats

ROOT     = Path('D:/Regional-Scale-AOD-Estimation')
LOSO_DIR = ROOT / 'modeling/results/loso'
SCATTER_DIR = LOSO_DIR / 'plots' / 'scatter'

for d in [LOSO_DIR / 'plots', SCATTER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

N_BOOT = 1000
N_PERM = 10_000
SEED   = 42

print('Dirs:', LOSO_DIR)

Dirs: D:\Regional-Scale-AOD-Estimation\modeling\results\loso


In [2]:
site_df = pd.read_csv(LOSO_DIR / 'per_site_metrics.csv')
pred_df = pd.read_csv(LOSO_DIR / 'loso_predictions.csv')

sites  = sorted(site_df['site'].unique())
models = sorted(site_df['model'].unique())
print(f'Sites: {len(sites)}')
print(f'Models: {models}')
print(f'site_df: {site_df.shape}  |  pred_df: {pred_df.shape}')

Sites: 20
Models: ['catboost', 'extratrees', 'gbm', 'lasso', 'lgbm', 'rf', 'ridge', 'svr', 'tabpfn_mi300', 'tabpfn_pca150', 'xgboost']
site_df: (440, 11)  |  pred_df: (112948, 5)


In [3]:
def compute_metrics(y_true, y_pred):
    """Canonical AOD-retrieval metric panel (identical to 02_evaluate)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt, yp = y_true[mask], y_pred[mask]
    n = len(yt)
    if n < 3:
        return {k: np.nan for k in ['N','RMSE','MAE','MBE','RMB','Pearson_r',
                                     'Spearman_r','R2','slope','intercept',
                                     'pct_within_EE','pct_above_EE','pct_below_EE']}
    residuals = yp - yt
    rmse = np.sqrt(np.mean(residuals ** 2))
    mae  = np.mean(np.abs(residuals))
    mbe  = np.mean(residuals)
    rmb  = np.mean(yp) / np.mean(yt) if np.mean(yt) != 0 else np.nan
    pearson_r, _ = stats.pearsonr(yt, yp)
    spearman_r   = stats.spearmanr(yt, yp).statistic
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((yt - yt.mean()) ** 2)
    r2     = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    slope, intercept, *_ = stats.linregress(yt, yp)
    # Levy et al. (2013) Dark Target land EE envelope: ±(0.05 + 0.15·AOD)
    EE = 0.05 + 0.15 * yt
    pct_within = np.mean(np.abs(residuals) <= EE) * 100
    pct_above  = np.mean(residuals > EE)  * 100
    pct_below  = np.mean(-residuals > EE) * 100
    return {
        'N': n, 'RMSE': rmse, 'MAE': mae, 'MBE': mbe, 'RMB': rmb,
        'Pearson_r': pearson_r, 'Spearman_r': spearman_r, 'R2': r2,
        'slope': slope, 'intercept': intercept,
        'pct_within_EE': pct_within, 'pct_above_EE': pct_above, 'pct_below_EE': pct_below,
    }

print('compute_metrics ready')

compute_metrics ready


In [4]:
# Cluster bootstrap CIs — resample SITES (n=20) with replacement, recompute pooled metric.
# This is the correct CI for LOSO: sites are the unit of spatial generalisation.

boot_rng = np.random.default_rng(SEED)

# Best transform per model (lowest pooled RMSE)
pooled_rows = []
for (model, transform), grp in pred_df.groupby(['model', 'transform']):
    m = compute_metrics(grp['y_true'], grp['y_pred'])
    pooled_rows.append({'model': model, 'transform': transform, **m})
pooled_df = pd.DataFrame(pooled_rows)

best_transform = (
    pooled_df.sort_values('RMSE')
    .groupby('model', as_index=False).first()[['model', 'transform']]
)

ci_rows = []
sites_arr = np.array(sites)

for _, row in best_transform.iterrows():
    model, transform = row['model'], row['transform']
    grp = pred_df[(pred_df['model'] == model) & (pred_df['transform'] == transform)]

    # Index predictions by site
    site_data = {s: grp[grp['site'] == s] for s in sites}

    boot_metrics = []
    for _ in range(N_BOOT):
        sampled_sites = boot_rng.choice(sites_arr, size=len(sites_arr), replace=True)
        boot_grp = pd.concat([site_data[s] for s in sampled_sites], ignore_index=True)
        boot_metrics.append(compute_metrics(boot_grp['y_true'], boot_grp['y_pred']))

    boot_df = pd.DataFrame(boot_metrics)
    point   = compute_metrics(grp['y_true'], grp['y_pred'])

    entry = {'model': model, 'transform': transform}
    entry.update(point)
    for col in boot_df.columns:
        if col == 'N':
            continue
        entry[f'{col}_ci_lo'] = float(np.nanpercentile(boot_df[col], 2.5))
        entry[f'{col}_ci_hi'] = float(np.nanpercentile(boot_df[col], 97.5))
    ci_rows.append(entry)

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(LOSO_DIR / 'loso_cluster_bootstrap_ci.csv', index=False)
print(f'Saved loso_cluster_bootstrap_ci.csv ({len(ci_df)} rows)')

display_cols = ['model', 'transform', 'N', 'RMSE', 'RMSE_ci_lo', 'RMSE_ci_hi',
                'MAE', 'Pearson_r', 'R2', 'pct_within_EE']
print('\n=== LOSO Pooled Metrics — Cluster Bootstrap 95% CIs ===')
print(ci_df.sort_values('RMSE')[display_cols].to_string(index=False))

Saved loso_cluster_bootstrap_ci.csv (11 rows)

=== LOSO Pooled Metrics — Cluster Bootstrap 95% CIs ===
        model transform    N     RMSE  RMSE_ci_lo  RMSE_ci_hi      MAE  Pearson_r        R2  pct_within_EE
        lasso     log1p 5134 0.116621    0.092498    0.144446 0.087895   0.720754  0.499716      57.713284
tabpfn_pca150     log1p 5134 0.126029    0.092407    0.164303 0.085259   0.699443  0.415744      62.504869
          svr     log1p 5134 0.126471    0.088724    0.165999 0.086413   0.666584  0.411638      61.453058
        ridge     log1p 5134 0.128441    0.099399    0.157011 0.096824   0.674145  0.393165      54.752630
          gbm     log1p 5134 0.131012    0.092987    0.182949 0.085285   0.665493  0.368629      62.056876
      xgboost     log1p 5134 0.138130    0.090811    0.206531 0.086168   0.639449  0.298162      63.011297
     catboost     log1p 5134 0.139951    0.091689    0.210256 0.087483   0.650065  0.279532      61.316712
 tabpfn_mi300     log1p 5134 0.148304    

In [5]:
# Scatter plots: one PNG per (model, transform) for LOSO pooled predictions
# Metric annotations from ci_df (cluster-bootstrap point estimates)
# Saved to results/loso/plots/scatter/{model}_{transform}.png

tf_map = best_transform.set_index('model')['transform'].to_dict()

for _, ci_row in ci_df.iterrows():
    model_name = ci_row['model']
    transform  = ci_row['transform']

    grp = pred_df[(pred_df['model'] == model_name) & (pred_df['transform'] == transform)]
    yt  = grp['y_true'].values
    yp  = grp['y_pred'].values

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(yt, yp, s=8, alpha=0.45, color='steelblue', rasterized=True)

    lim = max(yt.max(), yp.max()) * 1.06
    ax.plot([0, lim], [0, lim], 'k--', lw=1, label='1:1 line')

    t_range = np.linspace(0, lim, 300)
    ax.fill_between(t_range,
                    t_range - (0.05 + 0.15 * t_range),
                    t_range + (0.05 + 0.15 * t_range),
                    alpha=0.15, color='orange', label='EE envelope')

    stats_text = (
        f'N = {int(ci_row["N"])}\n'
        f'RMSE = {ci_row["RMSE"]:.4f}\n'
        f'MAE  = {ci_row["MAE"]:.4f}\n'
        f'MBE  = {ci_row["MBE"]:+.4f}\n'
        f'r    = {ci_row["Pearson_r"]:.3f}\n'
        f'R²   = {ci_row["R2"]:.3f}\n'
        f'%EE  = {ci_row["pct_within_EE"]:.1f}%'
    )
    ax.text(0.03, 0.97, stats_text,
            transform=ax.transAxes, fontsize=8,
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel('AOD true (550 nm)', fontsize=10)
    ax.set_ylabel('AOD predicted (550 nm)', fontsize=10)
    ax.set_title(f'{model_name} [{transform}] — LOSO', fontsize=10)
    ax.legend(fontsize=8, loc='lower right')
    ax.set_aspect('equal', adjustable='box')

    fig.savefig(SCATTER_DIR / f'{model_name}_{transform}.png', dpi=120, bbox_inches='tight')
    plt.close(fig)

n_files = len(list(SCATTER_DIR.glob('*.png')))
print(f'Saved {n_files} scatter PNGs → {SCATTER_DIR.relative_to(ROOT)}')

Saved 11 scatter PNGs → modeling\results\loso\plots\scatter


In [6]:
# Site-paired permutation test — compare per-site RMSE vectors (n=20 paired values)
# More appropriate for LOSO than sample-level permutation; sites are independent units.

perm_rng = np.random.default_rng(SEED)

# Per-site RMSE for best transform per model
site_rmse = {}   # model → array of 20 per-site RMSEs (aligned to sites_arr)
for _, row in best_transform.iterrows():
    model, transform = row['model'], row['transform']
    sub = site_df[(site_df['model'] == model) & (site_df['transform'] == transform)]
    sub = sub.set_index('site').reindex(sites)  # align to common site order
    site_rmse[model] = sub['RMSE'].values

perm_rows = []
model_list = sorted(site_rmse.keys())
print(f'Site-paired permutation: {len(model_list)} models, {len(list(combinations(model_list,2)))} pairs ...')

for m_a, m_b in combinations(model_list, 2):
    ra, rb = site_rmse[m_a], site_rmse[m_b]
    # Drop folds where either model has NaN RMSE (site with < 3 samples)
    valid = np.isfinite(ra) & np.isfinite(rb)
    if valid.sum() < 3:
        continue
    ra, rb = ra[valid], rb[valid]
    n = len(ra)

    diff = ra - rb
    obs_stat = np.mean(diff)

    perm_stats = np.empty(N_PERM)
    for k in range(N_PERM):
        signs = perm_rng.choice([-1, 1], size=n)
        perm_stats[k] = np.mean(diff * signs)

    p_val = np.mean(np.abs(perm_stats) >= np.abs(obs_stat))
    perm_rows.append({
        'model_a': m_a, 'model_b': m_b,
        'mean_rmse_a': float(np.mean(ra)),
        'mean_rmse_b': float(np.mean(rb)),
        'delta_rmse': float(obs_stat),   # positive = A worse
        'n_valid_sites': int(n),
        'p_value': float(p_val),
        'significant_05': p_val < 0.05,
    })

perm_df = pd.DataFrame(perm_rows)
perm_df.to_csv(LOSO_DIR / 'loso_site_permutation.csv', index=False)
print(f'Saved loso_site_permutation.csv')
sig = perm_df[perm_df['significant_05']].sort_values('p_value')
print(f'Significant pairs (p < 0.05): {len(sig)}')
if len(sig):
    print(sig[['model_a','model_b','delta_rmse','p_value']].to_string(index=False))

Site-paired permutation: 11 models, 55 pairs ...
Saved loso_site_permutation.csv
Significant pairs (p < 0.05): 0


In [7]:
# Significance heatmap — site-paired permutation p-value matrix (models × models)
# Color tiers: p < 0.001 (dark green), p < 0.01 (green), p < 0.05 (yellow), ns (light grey)
# Cell text: significance stars + Δ RMSE (positive = row model has higher mean-site RMSE than col)

SIG_CMAP_BOUNDS = [0, 0.001, 0.01, 0.05, 1.01]
SIG_COLORS      = ['#1a7f37', '#2ea44f', '#f0c400', '#d0d7de']
sig_cmap        = mcolors.BoundaryNorm(SIG_CMAP_BOUNDS, len(SIG_COLORS))
sig_palette     = mcolors.ListedColormap(SIG_COLORS)

models_uniq = sorted(set(perm_df['model_a']) | set(perm_df['model_b']))
n = len(models_uniq)
idx = {m: i for i, m in enumerate(models_uniq)}

p_mat     = np.ones((n, n))
delta_mat = np.full((n, n), np.nan)

for _, row in perm_df.iterrows():
    i, j = idx[row['model_a']], idx[row['model_b']]
    p_mat[i, j]     = row['p_value']
    p_mat[j, i]     = row['p_value']
    delta_mat[i, j] =  row['delta_rmse']   # positive = row model worse
    delta_mat[j, i] = -row['delta_rmse']

fig, ax = plt.subplots(figsize=(max(8, n * 0.75), max(7, n * 0.7)))
ax.imshow(p_mat, cmap=sig_palette, norm=sig_cmap, aspect='auto')

for i in range(n):
    for j in range(n):
        if i == j:
            ax.text(j, i, '—', ha='center', va='center', fontsize=7, color='grey')
        else:
            p_val = p_mat[i, j]
            d_val = delta_mat[i, j]
            stars = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
            d_str = f'{d_val:+.4f}' if not np.isnan(d_val) else ''
            ax.text(j, i, f'{stars}\n{d_str}',
                    ha='center', va='center', fontsize=6,
                    color='white' if p_val < 0.05 else '#24292f')

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(models_uniq, rotation=40, ha='right', fontsize=8)
ax.set_yticklabels(models_uniq, fontsize=8)
ax.set_title(
    'LOSO Site-Paired Permutation — Pairwise Significance\n'
    'Δ RMSE = row − col  (positive: row worse, negative: row better)\n'
    '★★★ p<0.001  ★★ p<0.01  ★ p<0.05  ns p≥0.05',
    fontsize=9
)

from matplotlib.patches import Patch
legend_els = [
    Patch(color='#1a7f37', label='p < 0.001'),
    Patch(color='#2ea44f', label='p < 0.01'),
    Patch(color='#f0c400', label='p < 0.05'),
    Patch(color='#d0d7de', label='ns (p ≥ 0.05)'),
]
ax.legend(handles=legend_els, loc='upper right',
          bbox_to_anchor=(1.28, 1.0), fontsize=8)

plt.tight_layout()
fig.savefig(LOSO_DIR / 'loso_sig_heatmap.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('Saved loso_sig_heatmap.png')

Saved loso_sig_heatmap.png


In [8]:
# Per-bin heatmaps for LOSO: compute quintile bins from pooled pred_df, then plot
# MAE heatmap (YlOrRd) + MBE heatmap (RdBu_r diverging), models sorted by pooled RMSE
# Saved to results/loso/heatmap_mae_loso.png and heatmap_mbe_loso.png

model_order_loso = ci_df.sort_values('RMSE')['model'].tolist()

bin_rows = []
for (model_name, transform), grp in pred_df.groupby(['model', 'transform']):
    yt = grp['y_true'].values
    yp = grp['y_pred'].values
    quantiles = np.quantile(yt, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
    for b in range(5):
        lo, hi = quantiles[b], quantiles[b + 1]
        mask = (yt >= lo) & (yt < hi) if b < 4 else (yt >= lo)
        if mask.sum() == 0:
            continue
        bin_rows.append({
            'model': model_name, 'transform': transform,
            'bin_idx': b + 1,
            'aod_lo': round(lo, 3), 'aod_hi': round(hi, 3),
            'N': int(mask.sum()),
            'MAE': float(np.mean(np.abs(yp[mask] - yt[mask]))),
            'MBE': float(np.mean(yp[mask] - yt[mask])),
        })

loso_bin_df = pd.DataFrame(bin_rows)
loso_bin_df.to_csv(LOSO_DIR / 'loso_per_bin_mae_mbe.csv', index=False)
print(f'Saved loso_per_bin_mae_mbe.csv ({len(loso_bin_df)} rows)')

# Filter to best transform per model
loso_bin_df = loso_bin_df[
    loso_bin_df.apply(lambda r: r['transform'] == tf_map.get(r['model'], r['transform']), axis=1)
]

# Build bin labels
bin_labels = []
for b in sorted(loso_bin_df['bin_idx'].unique()):
    grp_b = loso_bin_df[loso_bin_df['bin_idx'] == b]
    lo = grp_b['aod_lo'].iloc[0]
    hi = grp_b['aod_hi'].iloc[0]
    bin_labels.append(f'Q{b}\n{lo:.3f}–{hi:.3f}')

n_models = len(model_order_loso)
n_bins   = len(bin_labels)

mae_mat = np.full((n_models, n_bins), np.nan)
mbe_mat = np.full((n_models, n_bins), np.nan)

for mi, model_name in enumerate(model_order_loso):
    for bi, b_idx in enumerate(sorted(loso_bin_df['bin_idx'].unique())):
        cell = loso_bin_df[(loso_bin_df['model'] == model_name) & (loso_bin_df['bin_idx'] == b_idx)]
        if not cell.empty:
            mae_mat[mi, bi] = cell['MAE'].values[0]
            mbe_mat[mi, bi] = cell['MBE'].values[0]

def _draw_heatmap_loso(mat, title, cmap, vmin, vmax, fmt, out_path, center=None):
    fig, ax = plt.subplots(figsize=(max(7, n_bins * 1.4), max(4, n_models * 0.5)))
    if center is not None:
        norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
        im = ax.imshow(mat, cmap=cmap, norm=norm, aspect='auto')
    else:
        im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')

    for i in range(n_models):
        for j in range(n_bins):
            val = mat[i, j]
            if not np.isnan(val):
                ax.text(j, i, fmt.format(val),
                        ha='center', va='center', fontsize=7, color='black')

    ax.set_xticks(range(n_bins))
    ax.set_xticklabels(bin_labels, fontsize=8)
    ax.set_yticks(range(n_models))
    ax.set_yticklabels(
        [f'{m} [{tf_map[m]}]' for m in model_order_loso],
        fontsize=8
    )
    ax.set_xlabel('AOD quintile bin (true values)', fontsize=9)
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    plt.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out_path.name}')

_draw_heatmap_loso(
    mae_mat,
    title='LOSO Per-Bin MAE\n(lower = better; models sorted by pooled RMSE)',
    cmap='YlOrRd', vmin=0, vmax=np.nanmax(mae_mat), center=None,
    fmt='{:.4f}',
    out_path=LOSO_DIR / 'heatmap_mae_loso.png'
)

mbe_abs_max = np.nanmax(np.abs(mbe_mat))
_draw_heatmap_loso(
    mbe_mat,
    title='LOSO Per-Bin MBE\n(blue = under-predict, red = over-predict; models sorted by pooled RMSE)',
    cmap='RdBu_r', vmin=-mbe_abs_max, vmax=mbe_abs_max, center=0,
    fmt='{:+.4f}',
    out_path=LOSO_DIR / 'heatmap_mbe_loso.png'
)

Saved loso_per_bin_mae_mbe.csv (110 rows)
Saved: heatmap_mae_loso.png
Saved: heatmap_mbe_loso.png


In [9]:
# Per-site forest plot — RMSE per site for each model (best transform), models as rows
# Sites on x-axis, sorted by mean RMSE across all models.

model_order = ci_df.sort_values('RMSE')['model'].tolist()
site_mean_rmse = (
    site_df[site_df['model'].isin(model_list)]
    .groupby('site')['RMSE'].mean()
    .sort_values()
)
site_order = site_mean_rmse.index.tolist()

n_models = len(model_order)
fig, axes = plt.subplots(n_models, 1,
                          figsize=(max(14, len(site_order) * 0.7), n_models * 1.8),
                          sharex=True)
if n_models == 1:
    axes = [axes]

colors = plt.cm.tab10(np.linspace(0, 1, n_models))

for ax, model_name, color in zip(axes, model_order, colors):
    transform = best_transform.set_index('model').loc[model_name, 'transform']
    sub = site_df[
        (site_df['model'] == model_name) &
        (site_df['transform'] == transform)
    ].set_index('site').reindex(site_order)

    x = np.arange(len(site_order))
    ax.bar(x, sub['RMSE'].values, color=color, alpha=0.75, width=0.6)
    ax.axhline(ci_df[ci_df['model'] == model_name]['RMSE'].values[0],
               color='black', lw=1.2, ls='--', label='pooled RMSE')
    ax.set_ylabel('RMSE', fontsize=7)
    ax.set_title(f'{model_name} ({transform})', fontsize=8, loc='left')
    ax.tick_params(axis='y', labelsize=6)

axes[-1].set_xticks(np.arange(len(site_order)))
axes[-1].set_xticklabels(site_order, rotation=45, ha='right', fontsize=7)
plt.suptitle('LOSO Per-Site RMSE by Model (sites sorted by mean RMSE)', fontsize=11)
plt.tight_layout()
fig.savefig(LOSO_DIR / 'loso_forest_plot.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('Saved loso_forest_plot.png')

Saved loso_forest_plot.png


In [10]:
# Summary bar chart: pooled RMSE ± cluster-bootstrap CI, all models
ci_sorted = ci_df.sort_values('RMSE')
rmse_lo = np.maximum(ci_sorted['RMSE'] - ci_sorted['RMSE_ci_lo'], 0)
rmse_hi = np.maximum(ci_sorted['RMSE_ci_hi'] - ci_sorted['RMSE'], 0)

colors = ['#1f77b4' if t == 'raw' else '#ff7f0e' for t in ci_sorted['transform']]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(ci_sorted['model'], ci_sorted['RMSE'], color=colors,
       yerr=[rmse_lo.values, rmse_hi.values], capsize=3,
       error_kw={'linewidth': 1})
ax.set_ylabel('Pooled RMSE (LOSO)')
ax.set_title('LOSO Pooled RMSE ± 95% Cluster-Bootstrap CI')
ax.set_xticklabels(ci_sorted['model'], rotation=35, ha='right')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#1f77b4', label='raw'),
                   Patch(color='#ff7f0e', label='log1p')], fontsize=9)
plt.tight_layout()
fig.savefig(LOSO_DIR / 'loso_rmse_ci_bar.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('Saved loso_rmse_ci_bar.png')
print('\n04_evaluate_loso complete.')

Saved loso_rmse_ci_bar.png

04_evaluate_loso complete.
